# 04 — SelfModifyingLayer

Walkthrough of `hope.layers.SelfModifyingLayer`.

Paper references: §8.1 Eq. 76-91 (self-referential Titans), Eq. 18 (Hebbian fast-weight update used inside the per-token loop).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from hope.layers import SelfModifyingLayer

tf.random.set_seed(0)
np.random.seed(0)


## Forward pass on a small sequence

The slow projections `W_k`, `W_v`, `W_q` are trainable; the fast weight `W_fast` is reset to zero at the start of every call and updated after each token.

In [ ]:
layer = SelfModifyingLayer(units=8, eta=0.5, alpha=0.9)
x = tf.random.normal((1, 16, 4))
y = layer(x)
print('input shape :', tuple(x.shape))
print('output shape:', tuple(y.shape))
print('fast-weight shape after call:', tuple(layer.last_fast.shape))


## Fast weight diverges with two different inputs

We feed two random sequences of equal shape and inspect `||W_fast||_F` after each call.

In [ ]:
x1 = tf.random.normal((1, 16, 4))
x2 = tf.random.normal((1, 16, 4))
_ = layer(x1)
fast_after_1 = layer.last_fast.numpy().copy()
_ = layer(x2)
fast_after_2 = layer.last_fast.numpy().copy()

norm_1 = np.linalg.norm(fast_after_1)
norm_2 = np.linalg.norm(fast_after_2)
diff = np.linalg.norm(fast_after_1 - fast_after_2)

print(f'||W_fast after x1|| = {norm_1:.4f}')
print(f'||W_fast after x2|| = {norm_2:.4f}')
print(f'||(after x1) - (after x2)|| = {diff:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(fast_after_1[0], cmap='RdBu', vmin=-1, vmax=1)
ax[0].set_title('W_fast after x1')
ax[1].imshow(fast_after_2[0], cmap='RdBu', vmin=-1, vmax=1)
ax[1].set_title('W_fast after x2')
for a in ax: a.set_axis_off()
fig.tight_layout()
plt.show()


That divergence is the "self-modifying" behaviour: the operator's effective parameters depend on the sequence it has just seen.